# Notebook 10 (Revised) — Classical Models
## Milestone 1: Canonical modelling-data parity and protocol guards

**Scientific reference:** executed Notebook 09, source at commit `369a9d9`.

This is a *new notebook*, not a continuation of the 124-cell historical `10_secondary_ml_models.ipynb`. The historical notebook and its results stay untouched.

This milestone builds **no fitted models, no test predictions, and no written result artifacts**. It establishes exact source parity, independent target/lag checks, sampled F1–F4 numerical/category parity, and the locked availability-safe CV masks. Later modelling will be added only after these assertions pass.


In [1]:
from pathlib import Path
import hashlib
import json

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
REFERENCE_NOTEBOOK = PROJECT_ROOT / "notebooks" / "09_modelling_dataset_and_protocol.ipynb"

# Canonical construction: the first FIVE code cells in Notebook 09 at
# validated project commit 369a9d9.  Do not permit silent code drift.
NOTEBOOK09_CODE_SHA256 = (
    "18d232021455524939be8e14fc67842819ea4b8701a41a279b8997a47768bbee",
    "6add41c3649cfd7300a1fa04e0a966f30f7fc78c2a4b4bbaa054056358ddc21b",
    "724b7cdf9cd01790c9a0c9371967ff1c264c0f0d9730d1ef8d9be30dd359456d",
    "303947573618f2bcc5c280e2dd6e2d392805dc194073d51a8e4c84c7b4f3a373",
    "58ab56827f81c210df0ec662208f217baa3043d22c1940b028b041407efa9f09",
)

if not REFERENCE_NOTEBOOK.is_file():
    raise FileNotFoundError(f"Cannot find canonical Notebook 09: {REFERENCE_NOTEBOOK}")
with REFERENCE_NOTEBOOK.open(encoding="utf-8") as handle:
    _reference = json.load(handle)
reference_code_cells = [
    "".join(cell.get("source", []))
    for cell in _reference["cells"]
    if cell["cell_type"] == "code"
]
actual_hashes = tuple(
    hashlib.sha256(source.encode("utf-8")).hexdigest()
    for source in reference_code_cells[:5]
)
if actual_hashes != NOTEBOOK09_CODE_SHA256:
    raise RuntimeError(
        "Notebook 09 canonical construction source changed since commit 369a9d9. "
        "Stop: review and resynchronize the revised Notebook 10 instead of "
        "training with mismatched feature definitions."
    )
print("Canonical Notebook 09 source hash parity: PASSED (5 construction cells)")
print("Notebook 09 source: read-only")


Canonical Notebook 09 source hash parity: PASSED (5 construction cells)
Notebook 09 source: read-only


## A. Canonical Notebook 09 construction

The next five code cells are **verbatim** from the finalized Notebook 09 construction section. The preflight SHA-256 guard stops on any unexpected source drift. Keeping these definitions identical is deliberate for Milestone 1; source-module parity is checked independently in Section B.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import swisseph as swe
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete
from src.preprocessing import (
    build_cleaned_solete_features,
    build_gregorian_features,
)
import src.astronomy_features as astronomy_features
import src.vedic_calendar as vedic_calendar
from src.astronomy_features import (
    build_lunar_f3_features,
    build_solar_f2_features,
    load_jpl_ephemeris,
    solete_astronomy_reference_times,
    tropical_longitudes_for_solete,
)
from src.vedic_calendar import (
    build_panchang_f4_raw,
    lahiri_ayanamsha_for_utc_times,
)
from src.feature_families import (
    encode_panchang_fixed_vocabulary,
    PANCHANG_INDICATOR_COUNT,
)

DATA_5MIN = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5"
EPHEMERIS_FILE = PROJECT_ROOT / "ephemeris" / "de440s.bsp"

for path in [DATA_5MIN, EPHEMERIS_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)

# ------------------------------------------------------------------
# Source-parity guard.
#
# Notebook 07 froze pvlib altitude=0 m and delta_t=None.
# Notebook 08 froze standard Lahiri with FLG_NONUT.
#
# Stop immediately if reusable source files still implement the older
# conventions. A source-sync helper is supplied alongside this notebook.
# ------------------------------------------------------------------

if not hasattr(
    astronomy_features,
    "SOLETE_SOLAR_ALTITUDE_CONVENTION_M",
):
    raise RuntimeError(
        "src/astronomy_features.py is older than revised Notebook 07. "
        "Apply the supplied source-sync helper before running Notebook 09."
    )

assert astronomy_features.SOLETE_SOLAR_ALTITUDE_CONVENTION_M == 0.0

assert (
    vedic_calendar.LAHIRI_AYANAMSHA_FLAGS
    & swe.FLG_NONUT
), (
    "src/vedic_calendar.py still uses the older nutation-included "
    "Lahiri convention. Apply the supplied source-sync helper first."
)

df_raw = load_solete(DATA_5MIN).sort_index()

assert df_raw.index.is_monotonic_increasing
assert not df_raw.index.has_duplicates
assert df_raw.index.tz is not None

expected_step = pd.Timedelta(minutes=5)
observed_steps = df_raw.index.to_series().diff().dropna()
assert (observed_steps == expected_step).all(), (
    "SOLETE modelling timeline is not a complete 5-minute grid."
)

print("SOLETE:", df_raw.shape)
print("Period:", df_raw.index.min(), "to", df_raw.index.max())
print("Astronomy offset:", astronomy_features.SOLETE_ASTRONOMY_OFFSET)
print("Solar altitude convention [m]:",
      astronomy_features.SOLETE_SOLAR_ALTITUDE_CONVENTION_M)
print("Lahiri flags:", vedic_calendar.LAHIRI_AYANAMSHA_FLAGS)


SOLETE: (131617, 11)
Period: 2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00
Astronomy offset: 0 days 00:02:00
Solar altitude convention [m]: 0.0
Lahiri flags: 66


In [3]:
RESOLUTION_MINUTES = 5
FORECAST_HORIZONS = {
    15: 3,
    30: 6,
    60: 12,
}

PRIMARY_TARGET = "P_Solar[kW]"

F0_HISTORY_SOURCES = {
    "P_Solar_kw": "P_Solar[kW]",
    "GHI_kw1m2": "GHI[kW1m2]",
    "POA_Irr_kw1m2": "POA Irr[kW1m2]",
    "temperature_degC": "TEMPERATURE[degC]",
    "humidity_clean": "Humidity_clean",
    "pressure_clean_mbar": "Pressure_clean[mbar]",
    "wind_speed_m1s": "WIND_SPEED[m1s]",
    "wind_direction_sin": "wind_direction_sin",
    "wind_direction_cos": "wind_direction_cos",
}

RECENT_LAGS = tuple(range(13))
DAILY_LAG_STEPS = 24 * 60 // RESOLUTION_MINUTES

assert DAILY_LAG_STEPS == 288

df_clean = build_cleaned_solete_features(df_raw)

required_f0_sources = list(F0_HISTORY_SOURCES.values())
assert not df_clean[required_f0_sources].isna().any().any()

# F0: nine channels at t,t-5,...,t-60 plus P_Solar(t-24h).
f0_columns = {}

for feature_name, source_column in F0_HISTORY_SOURCES.items():
    for lag in RECENT_LAGS:
        f0_columns[f"{feature_name}_lag_{lag}"] = (
            df_clean[source_column].shift(lag)
        )

f0_columns["P_Solar_kw_issue_minus_24h"] = (
    df_clean[PRIMARY_TARGET].shift(DAILY_LAG_STEPS)
)

df_f0_full = pd.DataFrame(f0_columns, index=df_clean.index)

assert df_f0_full.shape[1] == 118

first_complete_f0_issue = df_f0_full.dropna().index.min()
expected_first_complete_f0_issue = pd.Timestamp(
    "2018-06-02 00:00:00",
    tz="UTC",
)
assert first_complete_f0_issue == expected_first_complete_f0_issue

print("F0 full-timeline shape:", df_f0_full.shape)
print("First row with complete F0:", first_complete_f0_issue)


F0 full-timeline shape: (131617, 118)
First row with complete F0: 2018-06-02 00:00:00+00:00


In [4]:
STUDY_START = pd.Timestamp("2018-06-01 00:00:00", tz="UTC")
LOCKED_TEST_START = pd.Timestamp("2019-06-01 00:00:00", tz="UTC")
STUDY_END = pd.Timestamp("2019-09-01 00:00:00", tz="UTC")

base_modelling_tables = {}

for horizon_minutes, horizon_steps in FORECAST_HORIZONS.items():
    table = df_f0_full.copy()

    table["issue_time"] = table.index
    table["valid_time"] = (
        table["issue_time"]
        + pd.Timedelta(minutes=horizon_minutes)
    )
    table["target_P_Solar_kW"] = (
        df_clean[PRIMARY_TARGET].shift(-horizon_steps)
    )
    table["horizon_minutes"] = horizon_minutes

    usable = (
        table[df_f0_full.columns].notna().all(axis=1)
        & table["target_P_Solar_kW"].notna()
        & table["valid_time"].ge(STUDY_START)
        & table["valid_time"].lt(STUDY_END)
    )

    table = table.loc[usable].copy()

    table["final_partition"] = np.where(
        table["valid_time"] < LOCKED_TEST_START,
        "development",
        "locked_test",
    )

    # Direct target-alignment verification.
    sample_positions = np.linspace(
        0, len(table) - 1, 5, dtype=int
    )
    for position in sample_positions:
        valid_time = table["valid_time"].iloc[position]
        observed = table["target_P_Solar_kW"].iloc[position]
        expected = df_clean.loc[valid_time, PRIMARY_TARGET]
        assert np.isclose(observed, expected, atol=0.0, rtol=0.0)

    base_modelling_tables[horizon_minutes] = table

population_summary = []

for horizon, table in base_modelling_tables.items():
    population_summary.append({
        "horizon_minutes": horizon,
        "rows": len(table),
        "development_rows":
            int(table["final_partition"].eq("development").sum()),
        "locked_test_rows":
            int(table["final_partition"].eq("locked_test").sum()),
        "first_issue": table["issue_time"].min(),
        "first_valid": table["valid_time"].min(),
        "last_valid": table["valid_time"].max(),
    })

display(pd.DataFrame(population_summary))


,horizon_minutes,rows,development_rows,locked_test_rows,first_issue,first_valid,last_valid
0,15,131325,104829,26496,2018-06-02 00:00:00+00:00,2018-06-02 00:15:00+00:00,2019-08-31 23:55:00+00:00
1,30,131322,104826,26496,2018-06-02 00:00:00+00:00,2018-06-02 00:30:00+00:00,2019-08-31 23:55:00+00:00
2,60,131316,104820,26496,2018-06-02 00:00:00+00:00,2018-06-02 01:00:00+00:00,2019-08-31 23:55:00+00:00


In [5]:
ephemeris = load_jpl_ephemeris(EPHEMERIS_FILE)

# F1: no astronomical offset.
df_f1_increment_full = build_gregorian_features(df_clean.index)
assert df_f1_increment_full.shape == (len(df_clean), 15)

# F2/F3: module applies row timestamp + 2 min.
df_f2_increment_full = build_solar_f2_features(df_clean.index)
df_f3_increment_full = build_lunar_f3_features(
    df_clean.index,
    ephemeris,
)

assert df_f2_increment_full.shape == (len(df_clean), 5)
assert df_f3_increment_full.shape == (len(df_clean), 9)

# F4 parents at the same +2 min astronomical reference.
df_tropical_full = tropical_longitudes_for_solete(
    df_clean.index,
    ephemeris,
)

astro_times = solete_astronomy_reference_times(df_clean.index)

ayanamsha = lahiri_ayanamsha_for_utc_times(astro_times)

df_ayanamsha = pd.Series(
    ayanamsha,
    index=df_clean.index,
    name="lahiri_ayanamsha_nonut_deg",
)

# Canonical Notebook 08 first-point guard.
KNOWN_FIRST_LAHIRI_NONUT = 24.11431760121269

assert np.isclose(
    df_ayanamsha.iloc[0],
    KNOWN_FIRST_LAHIRI_NONUT,
    atol=1e-10,
    rtol=0.0,
)

df_f4_raw_full = build_panchang_f4_raw(
    sun_tropical_longitude_deg=
        df_tropical_full["sun_tropical_longitude_deg"],
    moon_tropical_longitude_deg=
        df_tropical_full["moon_tropical_longitude_deg"],
    ayanamsha_deg=df_ayanamsha,
)

assert df_f4_raw_full.shape == (len(df_clean), 5)
assert not df_f4_raw_full.isna().any().any()

first_f4 = df_f4_raw_full.iloc[0]
assert int(first_f4["tithi"]) == 18
assert first_f4["paksha"] == "Krishna"
assert first_f4["karana"] == "Vanija"
assert first_f4["nakshatra"] == "Mula"
assert first_f4["yoga"] == "Shubha"

print("F1 increment:", df_f1_increment_full.shape)
print("F2 increment:", df_f2_increment_full.shape)
print("F3 increment:", df_f3_increment_full.shape)
print("Raw F4:", df_f4_raw_full.shape)
print("First Lahiri NONUT:", df_ayanamsha.iloc[0])
print("Notebook 07/08 source parity: PASSED")


F1 increment: (131617, 15)
F2 increment: (131617, 5)
F3 increment: (131617, 9)
Raw F4: (131617, 5)
First Lahiri NONUT: 24.11431760121269
Notebook 07/08 source parity: PASSED


In [6]:
EXPECTED_FEATURE_COUNTS = {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4": 244,
}

assert PANCHANG_INDICATOR_COUNT == 97

feature_family_tables = {}
panchang_onehot_tables = {}

for horizon, base in base_modelling_tables.items():
    issue_index = base.index
    valid_index = pd.DatetimeIndex(
        base["valid_time"],
        name="valid_time",
    )

    f0 = df_f0_full.loc[issue_index].copy()

    f1_inc = df_f1_increment_full.loc[valid_index].copy()
    f1_inc.index = issue_index
    f1_inc = f1_inc.add_prefix("valid_")

    f2_inc = df_f2_increment_full.loc[valid_index].copy()
    f2_inc.index = issue_index
    f2_inc = f2_inc.add_prefix("valid_")

    f3_inc = df_f3_increment_full.loc[valid_index].copy()
    f3_inc.index = issue_index
    f3_inc = f3_inc.add_prefix("valid_")

    f4_raw = df_f4_raw_full.loc[valid_index].copy()
    f4_raw.index = issue_index
    f4_raw = f4_raw.add_prefix("valid_")

    f4_onehot = encode_panchang_fixed_vocabulary(f4_raw)

    assert f4_onehot.shape[1] == 97
    assert (f4_onehot.sum(axis=1) == 5).all()

    F0 = f0
    F1 = pd.concat([F0, f1_inc], axis=1)
    F2 = pd.concat([F1, f2_inc], axis=1)
    F3 = pd.concat([F2, f3_inc], axis=1)
    F4 = pd.concat([F3, f4_onehot], axis=1)

    families = {
        "F0": F0,
        "F1": F1,
        "F2": F2,
        "F3": F3,
        "F4": F4,
    }

    for family, table in families.items():
        assert table.index.equals(issue_index)
        assert table.shape[1] == EXPECTED_FEATURE_COUNTS[family]
        assert not table.isna().any().any()

    assert list(F1.columns[:118]) == list(F0.columns)
    assert list(F2.columns[:133]) == list(F1.columns)
    assert list(F3.columns[:138]) == list(F2.columns)
    assert list(F4.columns[:147]) == list(F3.columns)

    feature_family_tables[horizon] = families
    panchang_onehot_tables[horizon] = f4_onehot

print("Nested feature construction: PASSED")

for horizon, families in feature_family_tables.items():
    print(
        horizon,
        {name: table.shape for name, table in families.items()},
    )


Nested feature construction: PASSED
15 {'F0': (131325, 118), 'F1': (131325, 133), 'F2': (131325, 138), 'F3': (131325, 147), 'F4': (131325, 244)}
30 {'F0': (131322, 118), 'F1': (131322, 133), 'F2': (131322, 138), 'F3': (131322, 147), 'F4': (131322, 244)}
60 {'F0': (131316, 118), 'F1': (131316, 133), 'F2': (131316, 138), 'F3': (131316, 147), 'F4': (131316, 244)}


## B. Independent source-level parity

Compare the canonical Notebook 09 F0/targets against `src.forecasting_protocol` with **exact values and ordered columns**. The daily-lag alias is explicit. The `split` in the legacy `build_base_modelling_tables()` output is ignored: revised models will use the corrected CV mask API only.

In [7]:
# Cross-check Notebook 09's independently written F0 and future-target
# calculations against the tested reusable forecasting protocol.
from src.forecasting_protocol import (
    F0_FEATURE_COLUMNS,
    FORECAST_HORIZONS as SOURCE_FORECAST_HORIZONS,
    build_base_modelling_tables as source_build_base,
    build_f0_history as source_build_f0,
    build_forecast_targets as source_build_targets,
)

assert FORECAST_HORIZONS == SOURCE_FORECAST_HORIZONS
source_f0 = source_build_f0(df_clean)
assert tuple(source_f0.columns) == F0_FEATURE_COLUMNS

# One known schema alias: both expressions mean P_Solar(issue_time - 24h).
# Do not silently relabel any other features.
source_f0 = source_f0.rename(
    columns={"P_Solar_kw_lag_288": "P_Solar_kw_issue_minus_24h"}
)
pd.testing.assert_frame_equal(source_f0, df_f0_full, check_exact=True)

source_targets = source_build_targets(df_clean)
source_base_tables = source_build_base(df_clean)
for horizon, base in base_modelling_tables.items():
    other = source_base_tables[horizon]
    assert base.index.equals(other.index), f"{horizon}: sample index mismatch"
    # Ignore only the two mutually incompatible *legacy* split metadata columns;
    # revised models use availability-safe CV masks below.
    compare_columns = list(df_f0_full.columns) + [
        "issue_time", "valid_time", "horizon_minutes", "target_P_Solar_kW"
    ]
    other_comparable = other.rename(
        columns={"P_Solar_kw_lag_288": "P_Solar_kw_issue_minus_24h"}
    )[compare_columns]
    pd.testing.assert_frame_equal(
        base[compare_columns], other_comparable, check_exact=True
    )
    pd.testing.assert_series_equal(
        base["target_P_Solar_kW"],
        source_targets[horizon].loc[base.index, "target_P_Solar_kW"],
        check_exact=True,
    )
print("F0 (118 columns) exact numerical/source parity: PASSED")
print("All horizon issue times, valid times and targets: PASSED")


F0 (118 columns) exact numerical/source parity: PASSED
All horizon issue times, valid times and targets: PASSED


## C. Independent deterministic-feature probes

Recompute F1–F4 for representative and boundary-adjacent valid timestamps using the currently frozen astronomy and Panchang sources. Compare numerical increments to strict tolerances and categorical one-hot indicators exactly. These probes supplement—but do not replace—the full-timeline construction copied from Notebook 09.

In [8]:
# Independently RECOMPUTE deterministic increments at boundary-focused
# probe times. This detects wrong issue/valid alignment or category encoding
# without recomputing the full ephemeris three additional times.

def _compare_increment(actual, expected, *, discrete=False):
    pd.testing.assert_index_equal(actual.columns, expected.columns)
    assert actual.shape == expected.shape
    assert not actual.isna().any().any()
    assert not expected.isna().any().any()
    if discrete:
        pd.testing.assert_frame_equal(
            actual.reset_index(drop=True), expected.reset_index(drop=True),
            check_exact=True,
        )
    else:
        np.testing.assert_allclose(
            actual.to_numpy(dtype=np.float64),
            expected.to_numpy(dtype=np.float64),
            atol=1e-8, rtol=1e-10,
        )


_PROBE_BOUNDARIES = pd.DatetimeIndex([
    "2018-12-01 00:00:00+00:00",
    "2019-02-01 00:00:00+00:00",
    "2019-04-01 00:00:00+00:00",
    "2019-06-01 00:00:00+00:00",
    "2019-09-01 00:00:00+00:00",
])

probe_summary = []
for horizon, base in base_modelling_tables.items():
    all_valid = pd.DatetimeIndex(base["valid_time"])
    indices = {0, len(base) - 1}
    indices.update(np.linspace(0, len(base) - 1, num=17, dtype=int))
    for boundary in _PROBE_BOUNDARIES:
        position = int(all_valid.searchsorted(boundary))
        indices.update(p for p in (position - 1, position, position + 1)
                       if 0 <= p < len(base))
    positions = np.array(sorted(indices), dtype=int)
    issues = base.index[positions]
    valid = pd.DatetimeIndex(base["valid_time"].iloc[positions])
    families = feature_family_tables[horizon]

    greg = build_gregorian_features(valid).add_prefix("valid_")
    solar = build_solar_f2_features(valid).add_prefix("valid_")
    lunar = build_lunar_f3_features(valid, ephemeris).add_prefix("valid_")

    _compare_increment(
        families["F1"].loc[issues, greg.columns], greg
    )
    _compare_increment(
        families["F2"].loc[issues, solar.columns], solar
    )
    _compare_increment(
        families["F3"].loc[issues, lunar.columns], lunar
    )

    tropical = tropical_longitudes_for_solete(valid, ephemeris)
    astronomy_times = solete_astronomy_reference_times(valid)
    ayan = pd.Series(
        lahiri_ayanamsha_for_utc_times(astronomy_times),
        index=valid,
        name="lahiri_ayanamsha_nonut_deg",
    )
    panchang_raw = build_panchang_f4_raw(
        sun_tropical_longitude_deg=tropical["sun_tropical_longitude_deg"],
        moon_tropical_longitude_deg=tropical["moon_tropical_longitude_deg"],
        ayanamsha_deg=ayan,
    ).add_prefix("valid_")
    encoded = encode_panchang_fixed_vocabulary(panchang_raw)
    _compare_increment(
        families["F4"].loc[issues, encoded.columns], encoded,
        discrete=True,
    )
    assert (encoded.sum(axis=1) == 5).all()
    probe_summary.append({
        "horizon_minutes": horizon,
        "probe_rows": len(positions),
        "F1-F3_numerical_recompute": "PASS",
        "F4_exact_onehot_recompute": "PASS",
    })

display(pd.DataFrame(probe_summary))
print("Sampled F1-F4 re-computation and row alignment: PASSED")


,horizon_minutes,probe_rows,F1-F3_numerical_recompute,F4_exact_onehot_recompute
0,15,29,PASS,PASS
1,30,29,PASS,PASS
2,60,29,PASS,PASS


Sampled F1-F4 re-computation and row alignment: PASSED


## D. Corrected CV, final-refit and terminal-holdout masks

No model fit or terminal evaluation is permitted in this milestone. Verify all counts and `last training target-valid time < first validation issue time` conditions for the three folds. The 60-minute label-availability exclusion modifies masks, never the continuous observations or feature tables.

In [9]:
# Build masks solely from the finalized source protocol. The legacy
# base['split'] or notebook 10 split_labels must never control training.
from src.forecasting_protocol import (
    CV_FOLDS,
    EXPECTED_CV_COUNTS_BY_HORIZON,
    EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON,
    MODEL_AVAILABILITY_GAP,
    make_rolling_origin_masks,
    make_terminal_refit_mask,
    terminal_holdout_mask,
    validate_rolling_origin_counts,
)

assert MODEL_AVAILABILITY_GAP == pd.Timedelta(minutes=60)
assert len(CV_FOLDS) == 3

EXPECTED_TOTAL = {15: 131325, 30: 131322, 60: 131316}
EXPECTED_DEVELOPMENT = {15: 104829, 30: 104826, 60: 104820}
cv_masks = {}
refit_masks = {}
holdout_masks = {}
summary = []
for horizon, base in base_modelling_tables.items():
    assert len(base) == EXPECTED_TOTAL[horizon]
    valid = base["valid_time"]
    issue = base["issue_time"]
    holdout = terminal_holdout_mask(valid)
    refit = make_terminal_refit_mask(valid)
    folds = make_rolling_origin_masks(valid)

    assert int(holdout.sum()) == 26496
    assert int((~holdout).sum()) == EXPECTED_DEVELOPMENT[horizon]
    assert int(refit.sum()) == EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON[horizon]
    assert valid.loc[refit].max() < issue.loc[holdout].min()
    assert validate_rolling_origin_counts(valid, horizon) == (
        EXPECTED_CV_COUNTS_BY_HORIZON[horizon]
    )
    assert len(families := feature_family_tables[horizon]) == 5
    assert all(frame.index.equals(base.index) for frame in families.values())
    assert (base.index.to_series().diff().dropna() == pd.Timedelta(minutes=5)).all()

    cv_masks[horizon] = {}
    for spec, fold in zip(CV_FOLDS, folds, strict=True):
        train = fold["train_mask"]
        val = fold["validation_mask"]
        assert not (train & val).any()
        assert not ((train | val) & holdout).any()
        assert valid.loc[train].max() < issue.loc[val].min()
        assert valid.loc[val].max() < issue.loc[holdout].min()
        cv_masks[horizon][spec["fold"]] = {"train": train, "validation": val}
        summary.append({
            "horizon": horizon,
            "fold": spec["fold"],
            "train": int(train.sum()),
            "validation": int(val.sum()),
            "holdout": int(holdout.sum()),
        })
    refit_masks[horizon] = refit
    holdout_masks[horizon] = holdout

display(pd.DataFrame(summary))
print("Three-fold availability-safe protocol: PASSED")
print("Original observed timeline and all 26,496 terminal rows: PRESERVED")
print("Milestone 1 passed. No model has been fitted or scored.")


,horizon,fold,train,validation,holdout
0,15,1,52401,17856,26496
1,15,2,70257,16992,26496
2,15,3,87249,17556,26496
3,30,1,52398,17856,26496
4,30,2,70254,16992,26496
5,30,3,87246,17556,26496
6,60,1,52392,17856,26496
7,60,2,70248,16992,26496
8,60,3,87240,17556,26496


Three-fold availability-safe protocol: PASSED
Original observed timeline and all 26,496 terminal rows: PRESERVED
Milestone 1 passed. No model has been fitted or scored.


## Milestone 1 completion gate

Proceed to **Milestone 2: tested shared rolling-CV runner** only if every cell passed on the *real* HDF5 and DE440s files. The historical Notebook 10, historical result directories, and terminal-holdout observed targets must not be used for hyperparameter selection. This notebook holds the terminal mask only and does not compute holdout error metrics.

## E. Ridge F0 15-minute validation pilot — Milestone 2

This is a **development-only smoke test** of `src/revised_classical_cv.py`,
using the actual SOLETE F0 (118 features) at a 15-minute horizon. It fits the
five **frozen Ridge alpha configurations** independently on three rolling-origin
folds (15 fits). Configuration selection uses the **unweighted mean of three
fold MAEs**, with original grid order as the deterministic tie-break.

**Holdout safety:** Terminal-holdout target values are explicitly masked to
`NaN` in a temporary target copy before calling the runner. The pilot must not
read historical result files, score terminal test data, or persist results.
A successful pilot permits further CV implementation, **not** final test
assessment. Existing Notebook 09 feature/parity checks remain unchanged.


In [10]:
# Development-only Ridge smoke test: 1 horizon x 1 family x 5 alphas x 3 folds.
# Do not access the terminal targets, run final refits, or save result artifacts.
from src.classical_models import RIDGE_ALPHA_GRID, build_ridge_model
from src.revised_classical_cv import evaluate_solete_cv_candidates

PILOT_HORIZON = 15
PILOT_FAMILY = "F0"

pilot_base = base_modelling_tables[PILOT_HORIZON]
pilot_X = feature_family_tables[PILOT_HORIZON][PILOT_FAMILY]
assert pilot_X.index.equals(pilot_base.index)
assert pilot_X.shape == (EXPECTED_TOTAL[PILOT_HORIZON], 118)

# Avoid even exposing the terminal target values to model selection.
pilot_y = pilot_base["target_P_Solar_kW"].mask(
    holdout_masks[PILOT_HORIZON]
)
assert pilot_y.loc[holdout_masks[PILOT_HORIZON]].isna().all()
assert pilot_y.loc[~holdout_masks[PILOT_HORIZON]].notna().all()

ridge_pilot_grid = tuple({"alpha": float(a)} for a in RIDGE_ALPHA_GRID)
assert len(ridge_pilot_grid) == 5

ridge_pilot_fold_scores, ridge_pilot_ranking = evaluate_solete_cv_candidates(
    X=pilot_X,
    y=pilot_y,
    issue_time=pilot_base["issue_time"],
    valid_time=pilot_base["valid_time"],
    horizon_minutes=PILOT_HORIZON,
    configurations=ridge_pilot_grid,
    model_factory=lambda c: build_ridge_model(alpha=c["alpha"]),
)

assert len(ridge_pilot_fold_scores) == len(ridge_pilot_grid) * 3
assert len(ridge_pilot_ranking) == len(ridge_pilot_grid)
assert ridge_pilot_fold_scores.groupby("config_index")["fold"].nunique().eq(3).all()

# The recorded fold populations must match the canonical availability-safe masks.
fold_rows = (
    ridge_pilot_fold_scores.query("config_index == 0")
    .sort_values("fold")[["train_rows", "validation_rows"]]
    .to_numpy(dtype=int)
)
assert tuple(map(tuple, fold_rows)) == EXPECTED_CV_COUNTS_BY_HORIZON[PILOT_HORIZON]

# Confirm ranking uses equal fold weighting, not pooled validation rows.
means = ridge_pilot_fold_scores.groupby("config_index")["validation_mae"].mean()
for row in ridge_pilot_ranking.itertuples(index=False):
    assert np.isclose(row.cv_mae, means.loc[row.config_index], atol=1e-12, rtol=0)
assert ridge_pilot_ranking["rank"].tolist() == list(range(1, 6))

print("Ridge F0 15-min pilot: 15 development-only fits; three folds per alpha.")
print("No terminal targets used; no terminal evaluation or file writes.")
print("Per-fold candidate scores:")
display(ridge_pilot_fold_scores)
print("Validation ranking (lower mean CV MAE is better):")
display(ridge_pilot_ranking)
print("Milestone 2 Ridge pilot: PASSED")


Ridge F0 15-min pilot: 15 development-only fits; three folds per alpha.
No terminal targets used; no terminal evaluation or file writes.
Per-fold candidate scores:


,config_index,configuration,fold,train_rows,validation_rows,validation_mae,validation_rmse
0,0,"{""alpha"": 0.01}",1,52401,17856,0.156717,0.340077
1,0,"{""alpha"": 0.01}",2,70257,16992,0.212823,0.473554
2,0,"{""alpha"": 0.01}",3,87249,17556,0.301663,0.610517
3,1,"{""alpha"": 0.1}",1,52401,17856,0.156719,0.340079
4,1,"{""alpha"": 0.1}",2,70257,16992,0.212824,0.473554
5,1,"{""alpha"": 0.1}",3,87249,17556,0.301663,0.610517
6,2,"{""alpha"": 1.0}",1,52401,17856,0.156741,0.340095
7,2,"{""alpha"": 1.0}",2,70257,16992,0.212833,0.473554
8,2,"{""alpha"": 1.0}",3,87249,17556,0.301655,0.610520
9,3,"{""alpha"": 10.0}",1,52401,17856,0.156952,0.340257


Validation ranking (lower mean CV MAE is better):


,rank,config_index,cv_mae,cv_rmse,folds,configuration
0,1,0,0.223734,0.474716,3,"{""alpha"": 0.01}"
1,2,1,0.223735,0.474717,3,"{""alpha"": 0.1}"
2,3,2,0.223743,0.474723,3,"{""alpha"": 1.0}"
3,4,3,0.223820,0.474794,3,"{""alpha"": 10.0}"
4,5,4,0.224485,0.475614,3,"{""alpha"": 100.0}"


Milestone 2 Ridge pilot: PASSED


## F. Complete Ridge rolling-origin CV — development only

This section extends the verified F0/15-minute Ridge pilot to **all three forecast
horizons (15/30/60 minutes) and all five frozen feature families (F0–F4)**.

- Use the **five frozen Ridge alpha candidates** and exactly **three availability-safe
  chronological folds** for every horizon–family combination.
- Select each candidate using the **unweighted mean of the three fold validation MAEs**;
  break exact ties by the original grid order. Keep fold MAE and RMSE for audit.
- Reuse the verified 15-fit F0/15-minute pilot **in memory** rather than refitting
  that same condition; every other condition is evaluated afresh by the same runner.
- **Mask terminal target values before passing them to the runner.** Do not perform
  final refits, terminal predictions, significance tests or write result artifacts.

These results support development-only hyperparameter selection. Notebook 10 must not
interpret validation improvement as terminal test performance. Ridge uses the frozen
`StandardScaler -> Ridge` pipeline, fitted from scratch within each training fold;
the scaler transforms all model columns, including categorical indicators for F4.


In [11]:
# Complete Ridge development-only 3-horizon x 5-family rolling CV.
# No final refit, terminal prediction, metric, or file persistence is permitted.
from time import perf_counter

RIDGE_FAMILIES = ("F0", "F1", "F2", "F3", "F4")
RIDGE_HORIZONS = (15, 30, 60)
assert tuple(FORECAST_HORIZONS) == RIDGE_HORIZONS
assert tuple(EXPECTED_FEATURE_COUNTS) == RIDGE_FAMILIES
assert len(ridge_pilot_grid) == len(RIDGE_ALPHA_GRID) == 5
assert PILOT_HORIZON == 15 and PILOT_FAMILY == "F0"

ridge_all_fold_tables = []
ridge_all_rank_tables = []
ridge_cv_runtimes = []

for ridge_horizon in RIDGE_HORIZONS:
    ridge_base = base_modelling_tables[ridge_horizon]
    ridge_holdout = holdout_masks[ridge_horizon]
    assert ridge_holdout.index.equals(ridge_base.index)
    assert int(ridge_holdout.sum()) == 26496
    assert len(ridge_base) == EXPECTED_TOTAL[ridge_horizon]

    # The fitting/selection path receives missing holdout labels by design.
    ridge_dev_y = ridge_base["target_P_Solar_kW"].mask(ridge_holdout)
    assert ridge_dev_y.loc[ridge_holdout].isna().all()
    assert ridge_dev_y.loc[~ridge_holdout].notna().all()

    for ridge_family in RIDGE_FAMILIES:
        ridge_X = feature_family_tables[ridge_horizon][ridge_family]
        assert ridge_X.index.equals(ridge_base.index)
        assert ridge_X.shape == (
            EXPECTED_TOTAL[ridge_horizon], EXPECTED_FEATURE_COUNTS[ridge_family]
        )
        ridge_started = perf_counter()
        if (ridge_horizon, ridge_family) == (PILOT_HORIZON, PILOT_FAMILY):
            # Already computed with this same runner in the preceding code cell.
            ridge_fold = ridge_pilot_fold_scores.copy(deep=True)
            ridge_rank = ridge_pilot_ranking.copy(deep=True)
            ridge_reused = True
        else:
            ridge_fold, ridge_rank = evaluate_solete_cv_candidates(
                X=ridge_X,
                y=ridge_dev_y,
                issue_time=ridge_base["issue_time"],
                valid_time=ridge_base["valid_time"],
                horizon_minutes=ridge_horizon,
                configurations=ridge_pilot_grid,
                model_factory=lambda c: build_ridge_model(alpha=c["alpha"]),
            )
            ridge_reused = False

        assert len(ridge_fold) == 15 and len(ridge_rank) == 5
        assert sorted(ridge_fold["fold"].unique().tolist()) == [1, 2, 3]
        assert ridge_fold.groupby("config_index")["fold"].nunique().eq(3).all()
        assert ridge_rank["config_index"].is_unique
        assert ridge_rank["rank"].tolist() == [1, 2, 3, 4, 5]
        assert np.isfinite(ridge_fold[["validation_mae", "validation_rmse"]]).all().all()
        assert np.isfinite(ridge_rank[["cv_mae", "cv_rmse"]]).all().all()

        # Every candidate must have exactly the canonical training/validation rows.
        for ridge_candidate in range(len(ridge_pilot_grid)):
            candidate_rows = (
                ridge_fold.loc[ridge_fold["config_index"].eq(ridge_candidate)]
                .sort_values("fold")[["train_rows", "validation_rows"]]
                .to_numpy(dtype=int)
            )
            assert tuple(map(tuple, candidate_rows)) == (
                EXPECTED_CV_COUNTS_BY_HORIZON[ridge_horizon]
            )
        means = ridge_fold.groupby("config_index")["validation_mae"].mean()
        for ranked in ridge_rank.itertuples(index=False):
            assert np.isclose(ranked.cv_mae, means.loc[ranked.config_index],
                              rtol=0, atol=1e-12)

        # Keep all folds/candidates and the full ranked grid, not just winners.
        ridge_fold.insert(0, "feature_family", ridge_family)
        ridge_fold.insert(0, "horizon_minutes", ridge_horizon)
        ridge_rank.insert(0, "feature_family", ridge_family)
        ridge_rank.insert(0, "horizon_minutes", ridge_horizon)
        ridge_all_fold_tables.append(ridge_fold)
        ridge_all_rank_tables.append(ridge_rank)
        ridge_cv_runtimes.append({
            "horizon_minutes": ridge_horizon,
            "feature_family": ridge_family,
            "elapsed_seconds": perf_counter() - ridge_started,
            "reused_prior_pilot": ridge_reused,
        })
        winner = ridge_rank.iloc[0]
        alpha = ridge_pilot_grid[int(winner["config_index"])]["alpha"]
        print(f"Ridge {ridge_horizon:2d} min {ridge_family}: "
              f"alpha={alpha:g}, mean CV MAE={winner['cv_mae']:.6f} kW"
              + (" [prior pilot]" if ridge_reused else ""), flush=True)

ridge_cv_fold_scores = pd.concat(ridge_all_fold_tables, ignore_index=True)
ridge_cv_rankings = pd.concat(ridge_all_rank_tables, ignore_index=True)
ridge_cv_runtime_summary = pd.DataFrame(ridge_cv_runtimes)
ridge_cv_selected = (
    ridge_cv_rankings.loc[ridge_cv_rankings["rank"].eq(1)]
    .copy().reset_index(drop=True)
)
ridge_cv_selected["alpha"] = ridge_cv_selected["config_index"].map(
    lambda i: float(ridge_pilot_grid[int(i)]["alpha"])
)

assert len(ridge_cv_fold_scores) == 3 * 5 * 5 * 3 == 225
assert len(ridge_cv_rankings) == 3 * 5 * 5 == 75
assert len(ridge_cv_selected) == 3 * 5 == 15
assert len(ridge_cv_runtime_summary) == 15
assert ridge_cv_selected[["horizon_minutes", "feature_family"]].drop_duplicates().shape[0] == 15

# Reused F0/15 scores must be numerically identical to the verified pilot.
pd.testing.assert_frame_equal(
    ridge_cv_fold_scores.loc[
        ridge_cv_fold_scores["horizon_minutes"].eq(15)
        & ridge_cv_fold_scores["feature_family"].eq("F0")
    ].drop(columns=["horizon_minutes", "feature_family"]).reset_index(drop=True),
    ridge_pilot_fold_scores.reset_index(drop=True),
    check_exact=True,
)

print("\nSelected alpha and three-fold MAE for all 15 Ridge conditions:")
display(ridge_cv_selected[["horizon_minutes", "feature_family", "alpha",
                           "cv_mae", "cv_rmse", "config_index"]])
print("\nAll 225 per-fold candidate scores and 75 candidate rankings remain in memory.")
print("No terminal target values, predictions, final refits, or result writes were used.")
print("Complete Ridge rolling-CV selection: PASSED")


Ridge 15 min F0: alpha=0.01, mean CV MAE=0.223734 kW [prior pilot]
Ridge 15 min F1: alpha=100, mean CV MAE=0.306610 kW
Ridge 15 min F2: alpha=100, mean CV MAE=0.306517 kW
Ridge 15 min F3: alpha=0.01, mean CV MAE=0.264452 kW
Ridge 15 min F4: alpha=100, mean CV MAE=0.303371 kW
Ridge 30 min F0: alpha=0.01, mean CV MAE=0.301027 kW
Ridge 30 min F1: alpha=100, mean CV MAE=0.442702 kW
Ridge 30 min F2: alpha=100, mean CV MAE=0.443909 kW
Ridge 30 min F3: alpha=0.01, mean CV MAE=0.360826 kW
Ridge 30 min F4: alpha=100, mean CV MAE=0.444886 kW
Ridge 60 min F0: alpha=100, mean CV MAE=0.421255 kW
Ridge 60 min F1: alpha=100, mean CV MAE=0.669405 kW
Ridge 60 min F2: alpha=100, mean CV MAE=0.678198 kW
Ridge 60 min F3: alpha=0.01, mean CV MAE=0.508959 kW
Ridge 60 min F4: alpha=100, mean CV MAE=0.684748 kW

Selected alpha and three-fold MAE for all 15 Ridge conditions:


,horizon_minutes,feature_family,alpha,cv_mae,cv_rmse,config_index
0,15,F0,0.01,0.223734,0.474716,0
1,15,F1,100.00,0.306610,0.529709,4
2,15,F2,100.00,0.306517,0.528925,4
3,15,F3,0.01,0.264452,0.480674,0
4,15,F4,100.00,0.303371,0.521804,4
5,30,F0,0.01,0.301027,0.574842,0
6,30,F1,100.00,0.442702,0.690312,4
7,30,F2,100.00,0.443909,0.690326,4
8,30,F3,0.01,0.360826,0.584384,0
9,30,F4,100.00,0.444886,0.684663,4



All 225 per-fold candidate scores and 75 candidate rankings remain in memory.
No terminal target values, predictions, final refits, or result writes were used.
Complete Ridge rolling-CV selection: PASSED


## G. Ridge 60-minute calendar-feature diagnostic — development only

The full Ridge rolling-CV results show an unexpectedly large validation MAE increase
from F0 to F1/F4, especially at the 60-minute horizon. We must diagnose this
*before* interpreting it as an astronomical/Panchang effect.

This exploratory diagnostic **does not select, replace or retune** any of the
frozen F0–F4 configurations. It holds Ridge alpha fixed at **100** (the
already-selected value for both 60-minute F0 and F1) and compares four
predictor sets on the same three availability-safe folds:

1. **F0**: original 118 history features.
2. **F0 + basic calendar**: add only nine raw Gregorian fields.
3. **F0 + cyclic calendar**: add only six sine/cosine calendar fields.
4. **F1**: F0 plus all 15 Gregorian fields.

For each fold, compare training vs validation MAE, systematic prediction bias,
prediction extrema, and standardized coefficient magnitude. All four predictor
sets are evaluated on identical train/validation rows.

**Safeguards:** mask terminal targets before fitting, assert exact fold counts
and training-label availability, and cross-check F0/F1 diagnostic validation
MAEs against their existing frozen CV scores. This section makes no predictions
on the terminal holdout and writes no model results to disk.

The raw/cyclic calendar ablations are **diagnostics**, not additional primary
feature families or extra candidates added to the original grid. Because they
were motivated by observed development scores, they are exploratory.


In [12]:
# Development-only diagnostic: isolate raw vs cyclic Gregorian effects.
# This does not change Ridge CV selections or access terminal evaluation.
DIAG_HORIZON = 60
DIAG_ALPHA = 100.0
assert DIAG_HORIZON in RIDGE_HORIZONS
assert DIAG_ALPHA in RIDGE_ALPHA_GRID

for diag_family in ("F0", "F1"):
    selected = ridge_cv_selected.loc[
        ridge_cv_selected["horizon_minutes"].eq(DIAG_HORIZON)
        & ridge_cv_selected["feature_family"].eq(diag_family)
    ]
    assert len(selected) == 1 and float(selected.iloc[0]["alpha"]) == DIAG_ALPHA

diag_base = base_modelling_tables[DIAG_HORIZON]
diag_holdout = holdout_masks[DIAG_HORIZON]
diag_y = diag_base["target_P_Solar_kW"].mask(diag_holdout)
assert diag_y.loc[diag_holdout].isna().all()
assert diag_y.loc[~diag_holdout].notna().all()
assert int(diag_holdout.sum()) == 26496

f0_60 = feature_family_tables[DIAG_HORIZON]["F0"]
f1_60 = feature_family_tables[DIAG_HORIZON]["F1"]
assert f0_60.index.equals(f1_60.index) and f1_60.index.equals(diag_base.index)
assert f1_60.columns[:118].tolist() == f0_60.columns.tolist()

basic_calendar = (
    "valid_utc_year", "valid_utc_month", "valid_utc_day",
    "valid_utc_day_of_week", "valid_utc_day_of_year",
    "valid_utc_hour", "valid_utc_minute", "valid_utc_minute_of_day",
    "valid_utc_is_weekend",
)
cyclic_calendar = (
    "valid_utc_time_sin", "valid_utc_time_cos",
    "valid_utc_weekday_sin", "valid_utc_weekday_cos",
    "valid_utc_year_sin", "valid_utc_year_cos",
)
assert f1_60.columns[118:].tolist() == list(basic_calendar + cyclic_calendar)

diag_features = {
    "F0": f0_60,
    "F0_plus_basic": f1_60.loc[:, list(f0_60.columns) + list(basic_calendar)],
    "F0_plus_cyclic": f1_60.loc[:, list(f0_60.columns) + list(cyclic_calendar)],
    "F1": f1_60,
}
assert tuple(frame.shape[1] for frame in diag_features.values()) == (
    118, 127, 124, 133
)

ridge_calendar_diagnostic_records = []
ridge_calendar_coef_records = []
for variant, diag_X in diag_features.items():
    for fold_number in (1, 2, 3):
        masks = cv_masks[DIAG_HORIZON][fold_number]
        train = masks["train"]
        validation = masks["validation"]
        assert tuple(map(int, (train.sum(), validation.sum()))) == (
            EXPECTED_CV_COUNTS_BY_HORIZON[DIAG_HORIZON][fold_number - 1]
        )
        assert not ((train | validation) & diag_holdout).any()
        assert (diag_base.loc[train, "valid_time"].max()
                < diag_base.loc[validation, "issue_time"].min())
        train_X = diag_X.loc[train]
        val_X = diag_X.loc[validation]
        train_y = diag_y.loc[train].to_numpy(dtype=float)
        val_y = diag_y.loc[validation].to_numpy(dtype=float)
        assert np.isfinite(train_y).all() and np.isfinite(val_y).all()
        assert np.isfinite(train_X.to_numpy(dtype=float)).all()
        assert np.isfinite(val_X.to_numpy(dtype=float)).all()

        # A NEW preprocessing pipeline is fitted on this fold's training rows.
        model = build_ridge_model(alpha=DIAG_ALPHA)
        model.fit(train_X, train_y)
        train_pred = np.asarray(model.predict(train_X), dtype=float)
        val_pred = np.asarray(model.predict(val_X), dtype=float)
        assert np.isfinite(train_pred).all() and np.isfinite(val_pred).all()
        val_mae = float(np.mean(np.abs(val_y - val_pred)))
        ridge_calendar_diagnostic_records.append({
            "variant": variant,
            "fold": fold_number,
            "train_rows": int(train.sum()),
            "validation_rows": int(validation.sum()),
            "train_mae": float(np.mean(np.abs(train_y - train_pred))),
            "validation_mae": val_mae,
            "validation_bias_pred_minus_actual": float(np.mean(val_pred - val_y)),
            "pred_min": float(val_pred.min()),
            "pred_max": float(val_pred.max()),
            "fraction_negative_predictions": float(np.mean(val_pred < 0.0)),
        })
        coefs = model.named_steps["ridge"].coef_
        assert len(coefs) == diag_X.shape[1]
        for group, columns in (
            ("F0_history", f0_60.columns),
            ("basic_calendar", basic_calendar),
            ("cyclic_calendar", cyclic_calendar),
        ):
            present = [c for c in columns if c in diag_X.columns]
            if not present:
                continue
            indices = diag_X.columns.get_indexer(present)
            ridge_calendar_coef_records.append({
                "variant": variant,
                "fold": fold_number,
                "group": group,
                "features": len(present),
                "standardized_coef_l2": float(np.linalg.norm(coefs[indices])),
                "standardized_coef_max_abs": float(np.max(np.abs(coefs[indices]))),
            })

ridge_calendar_diagnostic_folds = pd.DataFrame(
    ridge_calendar_diagnostic_records
)
ridge_calendar_coefficient_summary = pd.DataFrame(ridge_calendar_coef_records)
assert len(ridge_calendar_diagnostic_folds) == 4 * 3 == 12
assert (ridge_calendar_diagnostic_folds.groupby("variant")["fold"]
        .nunique().eq(3)).all()

# Verify the diagnostic exactly reproduces F0/F1 prior CV MAEs at alpha=100.
for family in ("F0", "F1"):
    reference = ridge_cv_fold_scores.loc[
        ridge_cv_fold_scores["horizon_minutes"].eq(DIAG_HORIZON)
        & ridge_cv_fold_scores["feature_family"].eq(family)
        & ridge_cv_fold_scores["config_index"].eq(4)
    ].sort_values("fold")
    observed = ridge_calendar_diagnostic_folds.loc[
        ridge_calendar_diagnostic_folds["variant"].eq(family)
    ].sort_values("fold")
    assert reference["fold"].tolist() == observed["fold"].tolist()
    np.testing.assert_allclose(
        observed["validation_mae"].to_numpy(),
        reference["validation_mae"].to_numpy(),
        rtol=0.0, atol=1e-6,
    )

ridge_calendar_diagnostic_summary = (
    ridge_calendar_diagnostic_folds.groupby("variant", sort=False)
    .agg(mean_train_mae=("train_mae", "mean"),
         mean_validation_mae=("validation_mae", "mean"),
         mean_bias=("validation_bias_pred_minus_actual", "mean"))
    .reset_index()
)
print("60-minute Ridge alpha=100: same folds, four calendar feature variants.")
print("Three-fold development-only mean errors and bias (kW):")
display(ridge_calendar_diagnostic_summary)
print("Fold-level diagnostics, including prediction range:")
display(ridge_calendar_diagnostic_folds)
print("Standardized-coefficient group magnitudes (descriptive, not causal):")
display(ridge_calendar_coefficient_summary)
print("No terminal target evaluation or artifact writes occurred.")
print("Ridge calendar-feature diagnostic: PASSED")


60-minute Ridge alpha=100: same folds, four calendar feature variants.
Three-fold development-only mean errors and bias (kW):


,variant,mean_train_mae,mean_validation_mae,mean_bias
0,F0,0.413133,0.421255,0.066979
1,F0_plus_basic,0.421861,0.434860,0.028522
2,F0_plus_cyclic,0.435283,0.444482,0.014704
3,F1,0.430149,0.669405,0.212691


Fold-level diagnostics, including prediction range:


,variant,fold,train_rows,validation_rows,train_mae,validation_mae,validation_bias_pred_minus_actual,pred_min,pred_max,fraction_negative_predictions
0,F0,1,52392,17856,0.453857,0.344028,0.132443,-0.880428,4.498834,0.043739
1,F0,2,70248,16992,0.394446,0.406711,0.003524,-1.036211,6.209807,0.119645
2,F0,3,87240,17556,0.391095,0.513027,0.064971,-0.662558,7.508682,0.143541
3,F0_plus_basic,1,52392,17856,0.462549,0.377267,0.175637,-0.645642,4.606738,0.081821
4,F0_plus_basic,2,70248,16992,0.403865,0.405350,-0.071298,-0.923092,6.038171,0.244586
5,F0_plus_basic,3,87240,17556,0.399169,0.521964,-0.018773,-0.685351,7.396373,0.233254
6,F0_plus_cyclic,1,52392,17856,0.470264,0.375855,0.041634,-0.807574,4.387633,0.370128
7,F0_plus_cyclic,2,70248,16992,0.419914,0.434671,-0.080219,-0.712036,5.820578,0.341984
8,F0_plus_cyclic,3,87240,17556,0.415671,0.522920,0.082696,-0.461194,7.078403,0.197767
9,F1,1,52392,17856,0.461603,1.016093,0.749554,-0.907454,6.013823,0.273466


Standardized-coefficient group magnitudes (descriptive, not causal):


,variant,fold,group,features,standardized_coef_l2,standardized_coef_max_abs
0,F0,1,F0_history,118,0.850535,0.453279
1,F0,2,F0_history,118,0.862842,0.554879
2,F0,3,F0_history,118,0.947568,0.666855
3,F0_plus_basic,1,F0_history,118,0.834003,0.451083
4,F0_plus_basic,1,basic_calendar,9,0.078243,0.043377
5,F0_plus_basic,2,F0_history,118,0.838907,0.541162
6,F0_plus_basic,2,basic_calendar,9,0.146468,0.114618
7,F0_plus_basic,3,F0_history,118,0.917078,0.660246
8,F0_plus_basic,3,basic_calendar,9,0.121725,0.095967
9,F0_plus_cyclic,1,F0_history,118,0.837054,0.462551


No terminal target evaluation or artifact writes occurred.
Ridge calendar-feature diagnostic: PASSED


## H. Classical-tree runtime pilot — development only

**Purpose:** Verify that the frozen Random Forest and LightGBM estimators can fit
real SOLETE F4 data on this machine, estimate per-fit runtime and ensure the
shared CV runner handles their predictions without changing the scientific
protocol. This is a **resource smoke test**, **not** hyperparameter selection.

- **Horizon:** 60 minutes; **features:** F4 (244 columns, the widest family).
- **Fold:** the first availability-safe expanding-window fold only.
- **Configurations:** the first of the eight frozen candidate configurations
  for each architecture, in `src.classical_models` order.
- **Workload:** **two fits** total (one Random Forest and one LightGBM).

No terminal-holdout labels are supplied to the runner, no final refits are
performed, and this cell saves no artifacts. The single-fold MAEs are only
sanity checks; they must never be used for choosing a configuration. After
reviewing CPU/RAM behavior and runtime, we can run the complete 3-fold,
5-family, 3-horizon selection grids with all eight candidates.


In [13]:
# Development-only model feasibility pilot. Exactly two real-data fits.
import gc
from time import perf_counter

from src.classical_models import (
    RANDOM_FOREST_CONFIGS,
    LIGHTGBM_CONFIGS,
    build_random_forest_model,
    build_lightgbm_model,
)
from src.revised_classical_cv import evaluate_cv_candidates

TREE_PILOT_HORIZON = 60
TREE_PILOT_FAMILY = "F4"
TREE_PILOT_FOLD = 1
assert len(RANDOM_FOREST_CONFIGS) == len(LIGHTGBM_CONFIGS) == 8
assert EXPECTED_FEATURE_COUNTS[TREE_PILOT_FAMILY] == 244

tree_base = base_modelling_tables[TREE_PILOT_HORIZON]
tree_X = feature_family_tables[TREE_PILOT_HORIZON][TREE_PILOT_FAMILY]
tree_holdout = holdout_masks[TREE_PILOT_HORIZON]
tree_y = tree_base["target_P_Solar_kW"].mask(tree_holdout)
assert tree_X.index.equals(tree_base.index)
assert tree_X.shape == (131316, 244)
assert int(tree_holdout.sum()) == 26496
assert tree_y.loc[tree_holdout].isna().all()

# The same source masks as every revised-model CV evaluation.
tree_fold = make_rolling_origin_masks(tree_base["valid_time"])[0]
assert tree_fold["fold"] == TREE_PILOT_FOLD
assert (int(tree_fold["train_mask"].sum()),
        int(tree_fold["validation_mask"].sum())) == (52392, 17856)
assert not ((tree_fold["train_mask"] | tree_fold["validation_mask"])
            & tree_holdout).any()
assert (tree_base.loc[tree_fold["train_mask"], "valid_time"].max()
        < tree_base.loc[tree_fold["validation_mask"], "issue_time"].min())

tree_pilot_runs = (
    ("Random Forest", RANDOM_FOREST_CONFIGS[0], build_random_forest_model),
    ("LightGBM", LIGHTGBM_CONFIGS[0], build_lightgbm_model),
)
tree_pilot_results = []
for name, candidate, factory in tree_pilot_runs:
    started = perf_counter()
    per_fold, ranked = evaluate_cv_candidates(
        X=tree_X,
        y=tree_y,
        issue_time=tree_base["issue_time"],
        valid_time=tree_base["valid_time"],
        horizon_minutes=TREE_PILOT_HORIZON,
        folds=(tree_fold,),
        configurations=(dict(candidate),),
        model_factory=factory,
        forbidden_mask=tree_holdout,
    )
    elapsed = perf_counter() - started
    assert per_fold.shape[0] == ranked.shape[0] == 1
    observed = per_fold.iloc[0]
    assert int(observed["fold"]) == TREE_PILOT_FOLD
    assert (int(observed["train_rows"]),
            int(observed["validation_rows"])) == (52392, 17856)
    assert np.isfinite(observed["validation_mae"])
    assert np.isfinite(observed["validation_rmse"])
    tree_pilot_results.append({
        "model": name,
        "horizon_minutes": TREE_PILOT_HORIZON,
        "feature_family": TREE_PILOT_FAMILY,
        "fold": TREE_PILOT_FOLD,
        "configuration_index": 0,
        "train_rows": int(observed["train_rows"]),
        "validation_rows": int(observed["validation_rows"]),
        "fold_mae_kw": float(observed["validation_mae"]),
        "elapsed_seconds": float(elapsed),
    })
    print(f"{name}: single-fold fit + prediction finished in "
          f"{elapsed:.1f}s (validation MAE {observed['validation_mae']:.6f} kW)",
          flush=True)
    del per_fold, ranked
    gc.collect()

tree_pilot_summary = pd.DataFrame(tree_pilot_results)
assert len(tree_pilot_summary) == 2
assert tree_pilot_summary["elapsed_seconds"].gt(0).all()
display(tree_pilot_summary)
print("Tree-model resource pilot: PASSED (2 development-only fits)")
print("No full grid selection, terminal evaluation, or result writes performed.")


Random Forest: single-fold fit + prediction finished in 239.5s (validation MAE 0.195970 kW)
LightGBM: single-fold fit + prediction finished in 5.5s (validation MAE 0.176398 kW)


,model,horizon_minutes,feature_family,fold,configuration_index,train_rows,validation_rows,fold_mae_kw,elapsed_seconds
0,Random Forest,60,F4,1,0,52392,17856,0.195970,239.519783
1,LightGBM,60,F4,1,0,52392,17856,0.176398,5.476004


Tree-model resource pilot: PASSED (2 development-only fits)
No full grid selection, terminal evaluation, or result writes performed.


## I. Complete LightGBM rolling-origin CV — development only

**Purpose:** Complete the frozen LightGBM model-selection grid before beginning
resource-intensive Random Forest selection. The prior 60-minute/F4/Fold-1
LightGBM pilot passed (MAE 0.176398 kW; 3.05 seconds), but that isolated
score is **not** a selection result. This cell refits every candidate across
all three canonical expanding-window folds, five nested feature families, and
three predeclared forecast horizons.

- Horizons: **15, 30, 60 minutes** (forecast-valid-time evaluation).
- Feature families: **F0, F1, F2, F3, F4** (fixed feature definitions).
- Frozen candidates: **all 8** from `src.classical_models.LIGHTGBM_CONFIGS`.
- Folds: **3** availability-safe folds from `make_rolling_origin_masks`.
- Total workload: **360 real-data fits**, without reusing the pilot score.

The `target_P_Solar_kW` labels of the locked June–August 2019 holdout are
masked before being handed to the shared CV runner. Each run also passes the
holdout as `forbidden_mask` and checks the expected development-fold counts.
Ranks are based on **equally weighted fold MAE**, with RMSE as the
prespecified secondary ranking metric. CV selection is development-only:
**no terminal predictions, final refits, or result-file writes** occur.

The cell can take substantially longer than the two-fit pilot. It reports a
summary after each (horizon, family) block. Save the notebook after completion
so that the displayed selected configurations and runtimes are recorded.
Random Forest batching is a separate subsequent stage.


In [14]:
# Complete LightGBM development-only selection: 3 horizons x 5 families x 8 candidates x 3 folds.
# Uses the identical frozen configurations, feature tables, and CV runner as the pilot.
# IMPORTANT: no holdout scoring, final refits, or external result writes.
import gc
from time import perf_counter
from src.classical_models import LIGHTGBM_CONFIGS, build_lightgbm_model
from src.revised_classical_cv import evaluate_cv_candidates

LIGHTGBM_CV_HORIZONS = (15, 30, 60)
LIGHTGBM_CV_FAMILIES = ("F0", "F1", "F2", "F3", "F4")
lightgbm_grid = tuple(dict(config) for config in LIGHTGBM_CONFIGS)
assert tuple(FORECAST_HORIZONS) == LIGHTGBM_CV_HORIZONS
assert tuple(EXPECTED_FEATURE_COUNTS) == LIGHTGBM_CV_FAMILIES
assert len(lightgbm_grid) == 8
assert len({tuple(sorted(c.items())) for c in lightgbm_grid}) == 8
assert len(CV_FOLDS) == 3

lightgbm_all_fold_tables = []
lightgbm_all_rank_tables = []
lightgbm_cv_runtimes = []

for lgb_horizon in LIGHTGBM_CV_HORIZONS:
    lgb_base = base_modelling_tables[lgb_horizon]
    lgb_holdout = holdout_masks[lgb_horizon]
    assert lgb_base.index.equals(lgb_holdout.index)
    assert len(lgb_base) == EXPECTED_TOTAL[lgb_horizon]
    assert int(lgb_holdout.sum()) == 26496
    lgb_y = lgb_base["target_P_Solar_kW"].mask(lgb_holdout)
    assert lgb_y.loc[lgb_holdout].isna().all()
    assert lgb_y.loc[~lgb_holdout].notna().all()

    lgb_folds = make_rolling_origin_masks(lgb_base["valid_time"])
    assert len(lgb_folds) == 3
    for lgb_expected_fold, lgb_fold in enumerate(lgb_folds, 1):
        assert lgb_fold["fold"] == lgb_expected_fold
        train_mask = lgb_fold["train_mask"]
        validation_mask = lgb_fold["validation_mask"]
        assert not (train_mask & validation_mask).any()
        assert not ((train_mask | validation_mask) & lgb_holdout).any()
        assert lgb_base.loc[train_mask, "valid_time"].max() < (
            lgb_base.loc[validation_mask, "issue_time"].min()
        )
        assert (int(train_mask.sum()), int(validation_mask.sum())) == (
            EXPECTED_CV_COUNTS_BY_HORIZON[lgb_horizon][lgb_expected_fold - 1]
        )

    for lgb_family in LIGHTGBM_CV_FAMILIES:
        lgb_X = feature_family_tables[lgb_horizon][lgb_family]
        assert lgb_X.index.equals(lgb_base.index)
        assert lgb_X.shape == (
            EXPECTED_TOTAL[lgb_horizon], EXPECTED_FEATURE_COUNTS[lgb_family]
        )
        lgb_started = perf_counter()
        lgb_per_fold, lgb_ranked = evaluate_cv_candidates(
            X=lgb_X,
            y=lgb_y,
            issue_time=lgb_base["issue_time"],
            valid_time=lgb_base["valid_time"],
            horizon_minutes=lgb_horizon,
            folds=lgb_folds,
            configurations=lightgbm_grid,
            model_factory=build_lightgbm_model,
            forbidden_mask=lgb_holdout,
        )
        elapsed = perf_counter() - lgb_started
        assert len(lgb_per_fold) == 8 * 3 == 24
        assert len(lgb_ranked) == 8
        assert set(lgb_per_fold["config_index"]) == set(range(8))
        assert set(lgb_ranked["config_index"]) == set(range(8))
        assert sorted(lgb_per_fold["fold"].unique().tolist()) == [1, 2, 3]
        assert lgb_per_fold.groupby("config_index")["fold"].nunique().eq(3).all()
        assert not lgb_per_fold.duplicated(["config_index", "fold"]).any()
        assert lgb_ranked["rank"].tolist() == list(range(1, 9))
        assert np.isfinite(lgb_per_fold[["validation_mae", "validation_rmse"]]).all().all()
        assert np.isfinite(lgb_ranked[["cv_mae", "cv_rmse"]]).all().all()
        for lgb_config_index in range(8):
            observed_counts = (
                lgb_per_fold.loc[lgb_per_fold["config_index"].eq(lgb_config_index)]
                .sort_values("fold")[["train_rows", "validation_rows"]]
                .to_numpy(dtype=int)
            )
            assert tuple(map(tuple, observed_counts)) == (
                EXPECTED_CV_COUNTS_BY_HORIZON[lgb_horizon]
            )
        # Confirm the shared runner uses an UNWEIGHTED mean over the three folds.
        mean_mae = lgb_per_fold.groupby("config_index")["validation_mae"].mean()
        for ranked_row in lgb_ranked.itertuples(index=False):
            assert np.isclose(ranked_row.cv_mae, mean_mae.loc[ranked_row.config_index],
                              atol=1e-12, rtol=0)
        assert lgb_ranked["cv_mae"].is_monotonic_increasing

        lgb_per_fold.insert(0, "feature_family", lgb_family)
        lgb_per_fold.insert(0, "horizon_minutes", lgb_horizon)
        lgb_ranked.insert(0, "feature_family", lgb_family)
        lgb_ranked.insert(0, "horizon_minutes", lgb_horizon)
        lightgbm_all_fold_tables.append(lgb_per_fold)
        lightgbm_all_rank_tables.append(lgb_ranked)
        lightgbm_cv_runtimes.append({
            "horizon_minutes": lgb_horizon,
            "feature_family": lgb_family,
            "elapsed_seconds": float(elapsed),
            "fits": 24,
        })
        lgb_winner = lgb_ranked.iloc[0]
        print(f"LightGBM {lgb_horizon:2d} min {lgb_family}: "
              f"best configuration={int(lgb_winner['config_index'])}, "
              f"3-fold CV MAE={lgb_winner['cv_mae']:.6f} kW, "
              f"elapsed={elapsed:.1f}s (24 fits)", flush=True)
        gc.collect()

lightgbm_cv_fold_scores = pd.concat(lightgbm_all_fold_tables, ignore_index=True)
lightgbm_cv_rankings = pd.concat(lightgbm_all_rank_tables, ignore_index=True)
lightgbm_cv_runtime_summary = pd.DataFrame(lightgbm_cv_runtimes)
lightgbm_cv_selected = (
    lightgbm_cv_rankings.loc[lightgbm_cv_rankings["rank"].eq(1)]
    .copy().reset_index(drop=True)
)
assert len(lightgbm_cv_fold_scores) == 3 * 5 * 8 * 3 == 360
assert len(lightgbm_cv_rankings) == 3 * 5 * 8 == 120
assert len(lightgbm_cv_selected) == 3 * 5 == 15
assert len(lightgbm_cv_runtime_summary) == 15
assert lightgbm_cv_selected[["horizon_minutes", "feature_family"]].drop_duplicates().shape[0] == 15
assert lightgbm_cv_runtime_summary["fits"].sum() == 360

print("\nLightGBM: 15 selected configurations from COMPLETE 3-fold development CV:")
display(lightgbm_cv_selected[["horizon_minutes", "feature_family",
                              "config_index", "configuration", "cv_mae", "cv_rmse"]])
print("\nLightGBM elapsed time per 24-fit block:")
display(lightgbm_cv_runtime_summary)
print("All 360 fold scores and 120 candidate rankings remain in kernel memory.")
print("No terminal predictions, final refits, or result-file writes were performed.")
print("Complete LightGBM rolling-CV selection: PASSED")


LightGBM 15 min F0: best configuration=1, 3-fold CV MAE=0.186805 kW, elapsed=153.3s (24 fits)
LightGBM 15 min F1: best configuration=0, 3-fold CV MAE=0.177400 kW, elapsed=178.9s (24 fits)
LightGBM 15 min F2: best configuration=0, 3-fold CV MAE=0.178696 kW, elapsed=178.6s (24 fits)
LightGBM 15 min F3: best configuration=1, 3-fold CV MAE=0.180784 kW, elapsed=190.8s (24 fits)
LightGBM 15 min F4: best configuration=0, 3-fold CV MAE=0.179828 kW, elapsed=195.6s (24 fits)
LightGBM 30 min F0: best configuration=0, 3-fold CV MAE=0.243146 kW, elapsed=152.7s (24 fits)
LightGBM 30 min F1: best configuration=0, 3-fold CV MAE=0.220946 kW, elapsed=185.4s (24 fits)
LightGBM 30 min F2: best configuration=0, 3-fold CV MAE=0.219329 kW, elapsed=177.6s (24 fits)
LightGBM 30 min F3: best configuration=0, 3-fold CV MAE=0.221195 kW, elapsed=191.7s (24 fits)
LightGBM 30 min F4: best configuration=0, 3-fold CV MAE=0.222238 kW, elapsed=203.4s (24 fits)
LightGBM 60 min F0: best configuration=1, 3-fold CV MAE=0.32

,horizon_minutes,feature_family,config_index,configuration,cv_mae,cv_rmse
0,15,F0,1,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.186805,0.468321
1,15,F1,0,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.177400,0.451460
2,15,F2,0,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.178696,0.450115
3,15,F3,1,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.180784,0.452514
4,15,F4,0,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.179828,0.449686
5,30,F0,0,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.243146,0.553750
6,30,F1,0,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.220946,0.518293
7,30,F2,0,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.219329,0.517169
8,30,F3,0,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.221195,0.518283
9,30,F4,0,"{""learning_rate"": 0.03, ""n_estimators"": 300, ""...",0.222238,0.518961



LightGBM elapsed time per 24-fit block:


,horizon_minutes,feature_family,elapsed_seconds,fits
0,15,F0,153.251215,24
1,15,F1,178.894920,24
2,15,F2,178.638301,24
3,15,F3,190.812160,24
4,15,F4,195.629039,24
5,30,F0,152.690053,24
6,30,F1,185.398309,24
7,30,F2,177.592214,24
8,30,F3,191.657269,24
9,30,F4,203.435314,24


All 360 fold scores and 120 candidate rankings remain in kernel memory.
No terminal predictions, final refits, or result-file writes were performed.
Complete LightGBM rolling-CV selection: PASSED


## J. Archive complete LightGBM development-only CV metrics

This section saves **only aggregated/validation CV scores**, rankings, selected
configurations and runtimes (no predictions, raw targets, holdout scores or
model refits). It requires the complete Section I tables to be **in this
Python kernel's memory**. If the kernel was restarted, rerun Section I before
archiving; the displayed 15 winners do not reconstruct all 360 fold scores.

Outputs (created **only when this cell runs**) are written to
`results/development_only/revised_classical_cv_v1/lightgbm/`. Existing
files with different contents are never overwritten. Keep this archive out of
frozen historical result directories. The notebook outputs alone remain an
independent record of the 15 selected configurations.


In [15]:
# Archive already-completed LightGBM CV tables WITHOUT fitting any model.
import hashlib
from src.revised_rf_checkpoint import archive_lightgbm

required_lgb_tables = (
    "lightgbm_cv_fold_scores", "lightgbm_cv_rankings",
    "lightgbm_cv_selected", "lightgbm_cv_runtime_summary",
)
if not all(key in globals() for key in required_lgb_tables):
    raise RuntimeError(
        "LightGBM tables are not in this kernel. To archive all fold results, "
        "rerun Section I, then this archive cell. Do not reconstruct them "
        "from the displayed 15 winners. Random Forest can run separately."
    )
assert lightgbm_cv_fold_scores.shape[0] == 360
assert lightgbm_cv_rankings.shape[0] == 120
assert lightgbm_cv_selected.shape[0] == 15
assert lightgbm_cv_runtime_summary.shape[0] == 15
assert len(holdout_masks) == 3
assert all(int(holdout_masks[h].sum()) == 26496 for h in (15, 30, 60))
LGB_AUDIT_DIR = (
    PROJECT_ROOT / "results" / "development_only" /
    "revised_classical_cv_v1" / "lightgbm"
)
lgb_source_digest = hashlib.sha256(
    (PROJECT_ROOT / "src" / "classical_models.py").read_bytes()
    + (PROJECT_ROOT / "src" / "revised_classical_cv.py").read_bytes()
    + (PROJECT_ROOT / "src" / "forecasting_protocol.py").read_bytes()
).hexdigest()
lgb_manifest = archive_lightgbm(
    fold_scores=lightgbm_cv_fold_scores,
    rankings=lightgbm_cv_rankings,
    selected=lightgbm_cv_selected,
    runtimes=lightgbm_cv_runtime_summary,
    directory=LGB_AUDIT_DIR,
    source_sha256=lgb_source_digest,
)
print("LightGBM development CV archived: 360 fold scores, 120 rankings, 15 winners")
print("Archive:", LGB_AUDIT_DIR)
print("No model fits, raw labels, terminal scoring, or prediction writes.")


LightGBM development CV archived: 360 fold scores, 120 rankings, 15 winners
Archive: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/development_only/revised_classical_cv_v1/lightgbm
No model fits, raw labels, terminal scoring, or prediction writes.


## K. Random Forest complete CV — restart-safe development only

This section completes the **frozen 360-fit Random Forest development CV**:
15-, 30-, 60-minute horizons; F0–F4; eight configurations; three canonical
availability-safe folds. The first pilot configuration was **not** used for
model selection. A Random Forest fit is expensive, so each successful
**single candidate × fold** fit is atomically checkpointed to
`results/development_only/revised_classical_cv_v1/random_forest/`.

- **Per execution limit:** `RF_MAX_NEW_FITS = 12` (adjust between 1 and 24).
- **Resume:** Run this SAME cell again; it skips validated saved fits. If the
  kernel restarts, rerun the Notebook 09 construction/masks cells first.
- **Safety:** Checkpoints are tied to fingerprints of feature *values*,
  masked target values, fold masks, frozen candidate grid and model/CV code.
  Conflicting checkpoints fail closed; no automatic overwrite/reset.
- **Selection:** MAE averaged *equally over the three folds*, RMSE secondary.
  No block is ranked until all 24 fits are present. No hyperparameter tuning
  from the earlier single-fold pilot or interim checkpoints.
- **No terminal evaluation:** all 26,496 terminal targets are masked before
  the runner sees labels. No final refits or test predictions are performed.

A complete run needs repeated executions of this cell. Checkpoints are
**development validation metrics only**, deliberately distinct from historical
published-result directories. Avoid running two notebook kernels concurrently
against the same checkpoint files. If a process was force-killed, a stale
`.lock` file may require inspection before manually clearing it.


In [45]:
# Restart-safe Random Forest CV. RUN REPEATEDLY; checkpoint EVERY successful fit.
import gc
import hashlib
import pandas as pd
import numpy as np
from src.classical_models import RANDOM_FOREST_CONFIGS, build_random_forest_model
from src.revised_classical_cv import evaluate_cv_candidates
from src.revised_rf_checkpoint import run_block

RF_MAX_NEW_FITS = 12  # 1..24 NEW fits per execution; stop anytime between fits.
RF_HORIZONS = (15, 30, 60)
RF_FAMILIES = ("F0", "F1", "F2", "F3", "F4")
rf_grid = tuple(dict(c) for c in RANDOM_FOREST_CONFIGS)
assert 1 <= RF_MAX_NEW_FITS <= 24
assert len(rf_grid) == 8 and len({tuple(sorted(c.items())) for c in rf_grid}) == 8
assert tuple(FORECAST_HORIZONS) == RF_HORIZONS
assert tuple(EXPECTED_FEATURE_COUNTS) == RF_FAMILIES
assert len(CV_FOLDS) == 3
RF_CHECKPOINT_ROOT = (
    PROJECT_ROOT / "results" / "development_only" /
    "revised_classical_cv_v1" / "random_forest"
)
# Include the exact implementation files in checkpoint identity.
rf_source_files = (
    "src/classical_models.py", "src/revised_classical_cv.py",
    "src/forecasting_protocol.py", "src/revised_rf_checkpoint.py",
)
rf_source_digest = hashlib.sha256(b"".join(
    (PROJECT_ROOT / f).read_bytes() for f in rf_source_files
)).hexdigest()
rf_budget = RF_MAX_NEW_FITS
rf_checked = []
for rf_horizon in RF_HORIZONS:
    if rf_budget == 0:
        break
    rf_base = base_modelling_tables[rf_horizon]
    rf_holdout = holdout_masks[rf_horizon]
    assert len(rf_base) == EXPECTED_TOTAL[rf_horizon]
    assert int(rf_holdout.sum()) == 26496
    rf_y = rf_base["target_P_Solar_kW"].mask(rf_holdout)
    assert rf_y.loc[rf_holdout].isna().all()
    assert rf_y.loc[~rf_holdout].notna().all()
    rf_folds = make_rolling_origin_masks(rf_base["valid_time"])
    for rf_family in RF_FAMILIES:
        if rf_budget == 0:
            break
        rf_X = feature_family_tables[rf_horizon][rf_family]
        assert rf_X.index.equals(rf_base.index)
        assert rf_X.shape == (
            EXPECTED_TOTAL[rf_horizon], EXPECTED_FEATURE_COUNTS[rf_family]
        )
        rf_checkpoint_file = (
            RF_CHECKPOINT_ROOT / f"h{rf_horizon}_{rf_family}.json"
        )
        rf_fold, rf_rank, rf_new = run_block(
            X=rf_X,
            y=rf_y,
            issue_time=rf_base["issue_time"],
            valid_time=rf_base["valid_time"],
            forbidden_mask=rf_holdout,
            folds=rf_folds,
            configurations=rf_grid,
            model_factory=build_random_forest_model,
            evaluator=evaluate_cv_candidates,
            horizon_minutes=rf_horizon,
            feature_family=rf_family,
            expected_counts=EXPECTED_CV_COUNTS_BY_HORIZON[rf_horizon],
            checkpoint_path=rf_checkpoint_file,
            source_sha256=rf_source_digest,
            max_new_fits=rf_budget,
        )
        rf_budget -= rf_new
        rf_checked.append((rf_horizon, rf_family, rf_fold, rf_rank))
        print(f"RF progress: {rf_horizon} min {rf_family} "
              f"{len(rf_fold)}/24 fits; {rf_new} new fits this execution.", flush=True)
        gc.collect()

rf_complete = [(h, f, folds, ranks) for h, f, folds, ranks in rf_checked
               if ranks is not None]
rf_fit_count = sum(len(folds) for _, _, folds, _ in rf_checked)
print(f"RF this execution: {RF_MAX_NEW_FITS-rf_budget} new fits; "
      f"{len(rf_complete)} fully ranked blocks checked.")
if len(rf_checked) == 15 and rf_fit_count == 360:
    rf_cv_fold_scores = pd.concat([
        frame.assign(horizon_minutes=h, feature_family=f)
        for h,f,frame,_ in rf_checked
    ], ignore_index=True)
    rf_cv_rankings = pd.concat([
        frame.assign(horizon_minutes=h, feature_family=f)
        for h,f,_,frame in rf_checked
    ], ignore_index=True)
    rf_cv_selected = rf_cv_rankings.loc[
        rf_cv_rankings["rank"].eq(1)
    ].copy().reset_index(drop=True)
    assert len(rf_cv_fold_scores) == 360
    assert len(rf_cv_rankings) == 120
    assert len(rf_cv_selected) == 15
    display(rf_cv_selected[[
        "horizon_minutes", "feature_family", "config_index",
        "configuration", "cv_mae", "cv_rmse",
    ]])
    print("Complete Random Forest rolling-CV selection: PASSED")
    print("No terminal evaluation, predictions, or final refits performed.")
else:
    print("Random Forest development CV INCOMPLETE. Save notebook and rerun "
          "this cell to resume safely from per-fit checkpoints.")


RF progress: 15 min F0 24/24 fits; 0 new fits this execution.
RF progress: 15 min F1 24/24 fits; 0 new fits this execution.
RF progress: 15 min F2 24/24 fits; 0 new fits this execution.
RF progress: 15 min F3 24/24 fits; 0 new fits this execution.
RF progress: 15 min F4 24/24 fits; 0 new fits this execution.
RF progress: 30 min F0 24/24 fits; 0 new fits this execution.
RF progress: 30 min F1 24/24 fits; 0 new fits this execution.
RF progress: 30 min F2 24/24 fits; 0 new fits this execution.
RF progress: 30 min F3 24/24 fits; 0 new fits this execution.
RF progress: 30 min F4 24/24 fits; 0 new fits this execution.
RF progress: 60 min F0 24/24 fits; 0 new fits this execution.
RF progress: 60 min F1 24/24 fits; 0 new fits this execution.
RF progress: 60 min F2 24/24 fits; 0 new fits this execution.
RF progress: 60 min F3 24/24 fits; 0 new fits this execution.
RF 60 min F4 candidate 5/8 fold 1/3: MAE=0.203580 kW, 253.2s; saved 13/24
RF 60 min F4 candidate 5/8 fold 2/3: MAE=0.333959 kW, 421.

,horizon_minutes,feature_family,config_index,configuration,cv_mae,cv_rmse
0,15,F0,3,"{""max_depth"":30,""max_features"":0.5,""min_sample...",0.182358,0.463762
1,15,F1,2,"{""max_depth"":null,""max_features"":0.5,""min_samp...",0.179036,0.458757
2,15,F2,2,"{""max_depth"":null,""max_features"":0.5,""min_samp...",0.177618,0.455851
3,15,F3,2,"{""max_depth"":null,""max_features"":0.5,""min_samp...",0.179622,0.458195
4,15,F4,2,"{""max_depth"":null,""max_features"":0.5,""min_samp...",0.179598,0.458095
5,30,F0,7,"{""max_depth"":30,""max_features"":1.0,""min_sample...",0.237948,0.559967
6,30,F1,7,"{""max_depth"":30,""max_features"":1.0,""min_sample...",0.225092,0.540882
7,30,F2,7,"{""max_depth"":30,""max_features"":1.0,""min_sample...",0.226551,0.541282
8,30,F3,2,"{""max_depth"":null,""max_features"":0.5,""min_samp...",0.229083,0.538759
9,30,F4,2,"{""max_depth"":null,""max_features"":0.5,""min_samp...",0.227636,0.536350


Complete Random Forest rolling-CV selection: PASSED
No terminal evaluation, predictions, or final refits performed.
